# Break the Spam Filter

AWS AI/ML Week 3 hands-on task.

1. Train the 10-line spam detector from the slides on the SMS Spam Collection and note its accuracy.
2. Write 5 messages that try to fool it.
3. Record what it predicted and why it failed.

**Dataset:** [SMS Spam Collection (UCI)](https://archive.ics.uci.edu/dataset/228/sms+spam+collection), 5,574 labelled text messages ("ham" = not spam).
The data is not stored in this repo. The notebook downloads a tab-separated copy the first time it runs.

In [11]:
import csv
import os
import urllib.request

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

pd.set_option("display.max_colwidth", 80)

## 1. Load the data

In [12]:
DATA_URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
DATA_PATH = "data/sms.tsv"

os.makedirs("data", exist_ok=True)
if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

# quoting=csv.QUOTE_NONE: some messages contain a " character. With pandas' default quote
# handling that merges neighbouring messages and silently drops rows (5,572 instead of 5,574).
df = pd.read_csv(DATA_PATH, sep="\t", header=None, names=["label", "message"],
                 quoting=csv.QUOTE_NONE)
print(df.shape)
df.head()

(5574, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e bu..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA ...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives around here though"


In [13]:
print(df["label"].value_counts())
print("Spam share:", round((df["label"] == "spam").mean(), 3))
print("Duplicate rows:", df.duplicated().sum())

label
ham     4827
spam     747
Name: count, dtype: int64
Spam share: 0.134
Duplicate rows: 403


Only about 13% of the messages are spam. Keep that in mind when reading accuracy below.

**Limitation:** the 403 duplicate rows were kept, because the task uses the data as given. Identical messages can end up in both the training and the test set, which may make the test score look a little better than it would on genuinely new messages.

## 2. The 10-line model (slide 6)

Split first, turn words into numbers with TF-IDF, then `fit()` is the learning step.

In [14]:
emails = df["message"]
labels = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    emails, labels, test_size=0.2, random_state=42)

vec = TfidfVectorizer()              # text -> numbers (features)
model = MultinomialNB()              # a simple, fast classifier
model.fit(vec.fit_transform(X_train), y_train)   # learn

print("Trained on", len(X_train), "messages, held out", len(X_test), "for testing")

Trained on 4459 messages, held out 1115 for testing


## 3. Measure it (slide 10)

In [15]:
pred_tr = model.predict(vec.transform(X_train))
pred_te = model.predict(vec.transform(X_test))

print("train acc:", round(accuracy_score(y_train, pred_tr), 4))
print("test acc: ", round(accuracy_score(y_test, pred_te), 4))
print("precision:", round(precision_score(y_test, pred_te, pos_label="spam"), 4))
print("recall:   ", round(recall_score(y_test, pred_te, pos_label="spam"), 4))

print()
print("A model that always says 'ham' would score:", round((y_test == "ham").mean(), 4))

cm = confusion_matrix(y_test, pred_te, labels=["spam", "ham"])
pd.DataFrame(cm,
             index=["actually spam", "actually ham"],
             columns=["predicted spam", "predicted ham"])

train acc: 0.9733
test acc:  0.9596
precision: 1.0
recall:    0.7205

A model that always says 'ham' would score: 0.8556


,predicted spam,predicted ham
actually spam,116,45
actually ham,0,954


**Reading the numbers.** Test accuracy is about 96%, close to the train score (about 97%), so this is not overfitting.
Precision is perfect (it never flagged good mail in the test set), but recall is only about 72%:
it misses 45 of the 161 spam messages in the test set. A model that always says "ham" already scores about 86%,
so accuracy alone hides how many spam messages this model misses.

## 4. Break it: five messages

Three spam messages that try to slip through, and two normal messages that try to get flagged.

In [16]:
tests = [
    ("Cl@im your fr33 g1ft card now", "spam"),
    ("Y O U   W O N   F R E E   M O N E Y   C L I C K   H E R E", "spam"),
    ("Dear customer, your account has been selected for a reward. Kindly confirm your details at your earliest convenience.", "spam"),
    ("Reply STOP to cancel your bank statement alerts. Standard rates apply. Call customer care free of charge", "ham"),
    ("Mobile top up of 100 birr received. Your new balance is 250 birr. Thank you for using our service", "ham"),
]

messages = [m for m, _ in tests]
should_be = [s for _, s in tests]

features = vec.transform(messages)
spam_col = list(model.classes_).index("spam")

results = pd.DataFrame({
    "my message": messages,
    "model said": model.predict(features),
    "p(spam)": model.predict_proba(features)[:, spam_col].round(3),
    "should be": should_be,
})
results["fooled it?"] = results["model said"] != results["should be"]
results

,my message,model said,p(spam),should be,fooled it?
0,Cl@im your fr33 g1ft card now,ham,0.075,spam,True
1,Y O U W O N F R E E M O N E Y C L I C K H E R E,ham,0.131,spam,True
2,"Dear customer, your account has been selected for a reward. Kindly confirm y...",ham,0.202,spam,True
3,Reply STOP to cancel your bank statement alerts. Standard rates apply. Call ...,spam,0.820,ham,True
4,Mobile top up of 100 birr received. Your new balance is 250 birr. Thank you ...,spam,0.632,ham,True


## 5. Why did it fail?

For each message, look at what the model actually sees: which words are unknown, and which known words
push toward spam or toward ham. The "lean" is `log P(word | spam) - log P(word | ham)`:
positive leans spam, negative leans ham.

In [17]:
import numpy as np

classes = list(model.classes_)
spam_i, ham_i = classes.index("spam"), classes.index("ham")
analyzer = vec.build_analyzer()

print("Prior (before reading any words):",
      "ham =", round(float(np.exp(model.class_log_prior_[ham_i])), 3),
      "| spam =", round(float(np.exp(model.class_log_prior_[spam_i])), 3))
print()

def explain(message, top=6):
    tokens = analyzer(message)
    unknown = sorted({t for t in tokens if t not in vec.vocabulary_})
    lean = {
        t: float(model.feature_log_prob_[spam_i, vec.vocabulary_[t]]
                 - model.feature_log_prob_[ham_i, vec.vocabulary_[t]])
        for t in set(tokens) if t in vec.vocabulary_
    }
    ranked = sorted(lean.items(), key=lambda kv: kv[1], reverse=True)
    spammy = [(w, round(v, 2)) for w, v in ranked if v > 0][:top]
    hammy = [(w, round(v, 2)) for w, v in ranked[::-1] if v < 0][:top]

    print(message[:75])
    print("  words the model sees :", tokens if tokens else "NONE")
    print("  non-zero features    :", vec.transform([message]).nnz)
    print("  never seen in training:", unknown if unknown else "-")
    print("  leans spam           :", spammy if spammy else "-")
    print("  leans ham            :", hammy if hammy else "-")
    print()

for message in messages:
    explain(message)

Prior (before reading any words): ham = 0.869 | spam = 0.131

Cl@im your fr33 g1ft card now
  words the model sees : ['cl', 'im', 'your', 'fr33', 'g1ft', 'card', 'now']
  non-zero features    : 5
  never seen in training: ['fr33', 'g1ft']
  leans spam           : [('cl', 0.2)]
  leans ham            : [('im', -1.34), ('now', -0.27), ('card', -0.22), ('your', -0.17)]

Y O U   W O N   F R E E   M O N E Y   C L I C K   H E R E
  words the model sees : NONE
  non-zero features    : 0
  never seen in training: -
  leans spam           : -
  leans ham            : -

Dear customer, your account has been selected for a reward. Kindly confirm 
  words the model sees : ['dear', 'customer', 'your', 'account', 'has', 'been', 'selected', 'for', 'reward', 'kindly', 'confirm', 'your', 'details', 'at', 'your', 'earliest', 'convenience']
  non-zero features    : 14
  never seen in training: ['earliest']
  leans spam           : [('reward', 1.84), ('customer', 1.75), ('selected', 1.31), ('account', 0.7

In [18]:
# How common are some of these words in each class? (training data only)
train = pd.DataFrame({"message": X_train.str.lower(), "label": y_train})
for word in ["free", "stop", "customer"]:
    in_spam = train[train["label"] == "spam"]["message"].str.contains(rf"\b{word}\b").mean()
    in_ham = train[train["label"] == "ham"]["message"].str.contains(rf"\b{word}\b").mean()
    print(f"'{word}' appears in {in_spam:.1%} of spam and {in_ham:.1%} of ham")

'free' appears in 21.7% of spam and 1.3% of ham
'stop' appears in 13.5% of spam and 0.7% of ham
'customer' appears in 6.1% of spam and 0.1% of ham


**What the output shows**

1. `Cl@im your fr33 g1ft card now`: `fr33` and `g1ft` were never seen in training, and `Cl@im` is split into `cl` and `im`, where `im` is a strongly ham word.
2. `Y O U   W O N ...`: the default tokenizer ignores one-letter tokens, so the message has **zero** features. With nothing to read, the model falls back to its prior, which says ham.
3. `Dear customer, your account ...`: a few spam-leaning words (`customer`, `reward`, `selected`) are outweighed by ordinary words that lean ham (`dear`, `at`, `for`, `confirm`) and by the 87% ham prior.
4. `Reply STOP to cancel ... free of charge`: a real service message that reuses spam vocabulary (`apply`, `statement`, `customer`, `rates`, `standard`, `stop`). `free` is also there: it appears in 21.7% of spam but only 1.3% of ham (see the counts above). The model counts words and cannot tell who is sending or why.
5. `Mobile top up of 100 birr ...`: numbers and words like `mobile`, `service`, `balance` appear mostly in spam in this dataset, and `birr` was never seen.

### Tried, but did not fool it

The slide suggests negation ("Not free, just confirming lunch"). On this dataset the model gets it right.

In [19]:
negation = "Not free, just confirming lunch at 1pm"
print("Model said:", model.predict(vec.transform([negation]))[0],
      "| p(spam) =", round(float(model.predict_proba(vec.transform([negation]))[0][spam_col]), 3))
print()
explain(negation)

Model said: ham | p(spam) = 0.026

Not free, just confirming lunch at 1pm
  words the model sees : ['not', 'free', 'just', 'confirming', 'lunch', 'at', '1pm']
  non-zero features    : 6
  never seen in training: ['confirming']
  leans spam           : [('free', 1.33), ('1pm', 0.31)]
  leans ham            : [('not', -2.28), ('at', -2.18), ('lunch', -1.88), ('just', -0.97)]



It is not that the model understands "not". The ham-leaning words around it (`not`, `lunch`, `at`) are strong enough to outweigh `free`.

## 6. What I learned

A word-counting model only knows the words it saw in training: spelling tricks, spaced-out letters and polite wording hide spam from it, and ordinary service messages that reuse spam words get flagged. High accuracy hid this because most messages are not spam, so I should also look at precision and recall and test with messages written differently from the training data.